In [1]:
!git clone https://github.com/oscar2697/chess-transformer.git 2>/dev/null; true
%cd chess-transformer
!git pull origin main
!git log --oneline -2
!ls src/graph_agents/

/content/chess-transformer
From https://github.com/oscar2697/chess-transformer
 * branch            main       -> FETCH_HEAD
Already up to date.
85bc51b (HEAD -> main, origin/main, origin/HEAD) train: resume + AMP + val loop; preprocess multi-PGN; notebook kaggle limpio
9aacbb5 fix: enhance Streamlit UI with real-look chessboard and interactive features
graph.py  __init__.py  llm_agents.py  nodes  __pycache__


In [2]:
!pip install python-chess torch matplotlib langchain-openai -q
from src.graph_agents.llm_agents import run_agent, get_llm
print('import ok, backend:', get_llm()[0])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 76.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 44.8 MB/s eta 0:00:00
import ok, backend: None


In [4]:
from google.colab import userdata
import os
os.environ['LLM_PROVIDER'] = 'nvidia'
os.environ['NVIDIA_API_KEY'] = userdata.get('NVIDIA_KEY')
os.environ['LLM_MODEL'] = 'moonshotai/kimi-k3'
from src.graph_agents.llm_agents import get_llm
print('backend:', get_llm()[0])  # esperado: nvidia

backend: nvidia


In [5]:
!mkdir -p data/raw
!wget -q https://database.nikonoel.fr/lichess_elite_2024-01.zip -O data/raw/elite_2024-01.zip
!wget -q https://database.nikonoel.fr/lichess_elite_2024-02.zip -O data/raw/elite_2024-02.zip
!wget -q https://database.nikonoel.fr/lichess_elite_2024-03.zip -O data/raw/elite_2024-03.zip
!unzip -o -q data/raw/elite_2024-01.zip -d data/raw/
!unzip -o -q data/raw/elite_2024-02.zip -d data/raw/
!unzip -o -q data/raw/elite_2024-03.zip -d data/raw/
!ls -lh data/raw/*.pgn

-rw-r--r-- 1 root root 235M Feb  7  2024 data/raw/lichess_elite_2024-01.pgn
-rw-r--r-- 1 root root 202M Mar  5  2024 data/raw/lichess_elite_2024-02.pgn
-rw-rw-r-- 1 root root 252M May 21  2024 data/raw/lichess_elite_2024-03.pgn


In [6]:
from src.graph_agents.llm_agents import run_agent
r = run_agent('data_engineer', 'Preprocess 3 elite months',
              pgn_path=['data/raw/lichess_elite_2024-01.pgn',
                        'data/raw/lichess_elite_2024-02.pgn',
                        'data/raw/lichess_elite_2024-03.pgn'],
              elo_threshold=2000, max_positions=600000)
print('mode:', r['mode'])
print(r['result'])

/content/chess-transformer/src/data/pgn_parser.py:48: UserWarning: No PGN data for vocab; using deterministic enumeration slice. Run preprocess with real PGNs.
  warnings.warn("No PGN data for vocab; using deterministic enumeration slice. Run preprocess with real PGNs.")


Parsing /content/chess-transformer/data/raw/lichess_elite_2024-01.pgn ...
Preprocess -> {'n_train': 570000, 'n_val': 30000, 'dedup': 600000, 'source': ['/content/chess-transformer/data/raw/lichess_elite_2024-01.pgn', '/content/chess-transformer/data/raw/lichess_elite_2024-02.pgn', '/content/chess-transformer/data/raw/lichess_elite_2024-03.pgn']}
mode: llm:nvidia
{'n_train': 570000, 'n_val': 30000, 'dedup': 600000, 'source': ['/content/chess-transformer/data/raw/lichess_elite_2024-01.pgn', '/content/chess-transformer/data/raw/lichess_elite_2024-02.pgn', '/content/chess-transformer/data/raw/lichess_elite_2024-03.pgn']}


In [7]:
# CELDA RE-EJECUTABLE: si muere la sesión, corre de nuevo y continúa desde last.pt
from src.graph_agents.llm_agents import run_agent
r = run_agent('trainer', 'Scale training', epochs=15, batch_size=128, seed=42,
              resume=True, use_amp=True)
print('mode:', r['mode'])
print(r['result'])

Epoch 0 loss 7.6983 val 7.6061 lr 2.97e-04 (12.6 min)
Epoch 1 loss 7.6613 val 7.9627 lr 2.87e-04 (12.5 min)
Epoch 2 loss 7.6523 val 7.8086 lr 2.71e-04 (12.4 min)
Epoch 3 loss 7.6478 val 8.1271 lr 2.50e-04 (12.4 min)
Epoch 4 loss 7.6471 val 7.9155 lr 2.25e-04 (12.3 min)
Epoch 5 loss 7.6413 val 7.9864 lr 1.96e-04 (12.4 min)
Epoch 6 loss 7.6407 val 7.8837 lr 1.66e-04 (12.3 min)
Epoch 7 loss 7.6381 val 7.9997 lr 1.34e-04 (12.3 min)
Epoch 8 loss 7.6327 val 8.1022 lr 1.04e-04 (12.3 min)
Epoch 9 loss 7.6320 val 8.0325 lr 7.50e-05 (12.3 min)
Epoch 10 loss 7.6293 val 8.1106 lr 4.96e-05 (12.3 min)
Epoch 11 loss 7.6271 val 8.3733 lr 2.86e-05 (12.4 min)
Epoch 12 loss 7.6255 val 8.0882 lr 1.30e-05 (12.3 min)
Epoch 13 loss 7.6243 val 8.0685 lr 3.28e-06 (12.3 min)
Epoch 14 loss 7.6230 val 8.0625 lr 0.00e+00 (12.3 min)
mode: llm:nvidia
{'best_loss': 7.606123447418213, 'history': [{'epoch': 0, 'loss': 7.698278828913393, 'val_loss': 7.606123447418213, 'lr': 0.0002967221401100708, 'minutes': 12.6}, {'epo

In [8]:
from src.graph_agents.llm_agents import run_agent
print(run_agent('evaluator', 'Evaluate policy/value vs Stockfish')['result'])
print(run_agent('writer', 'Write results tables')['result'][:300])

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Eval -> /content/chess-transformer/experiments/evaluation_results.json: {'top1_accuracy': 0.0, 'top3_accuracy': 0.0, 'value_mse': 0.033719442818882124, 'n_positions': 3}
{'top1_accuracy': 0.0, 'top3_accuracy': 0.0, 'value_mse': 0.033719442818882124, 'n_positions': 3}
Paper patched -> /content/chess-transformer/paper/main.tex
\section{Results \& Evaluation}
\begin{table}[h]\centering\caption{Training Loss (CE+MSE)}\begin{tabular}{cc}\hline Epoch & Loss \\ \hline
0 & 7.6983 \\
1 & 7.6613 \\
2 & 7.6523 \\
3 & 7.6478 \\
4 & 7.6471 \\
5 & 7.6413 \\
6 & 7.6407 \\
7 & 7.6381 \\
8 & 7.6327 \\
9 & 7.6320 \\
10 & 7.6293 \\
11 & 7


In [9]:
!tar -czf artifacts.tar.gz checkpoints/best_model.pt checkpoints/last.pt experiments/ paper/main.tex data/processed/stats.json data/processed/vocab.json
!ls -lh artifacts.tar.gz
# Descarga artifacts.tar.gz + data/processed/train.jsonl + val.jsonl desde el panel Output

-rw-r--r-- 1 root root 363M Sep  8 20:28 artifacts.tar.gz


In [10]:
# 1. Localiza los archivos (quizá el cwd no es el repo)
!pwd
!find /content -maxdepth 4 -name "best_model.pt" 2>/dev/null
!find /content -maxdepth 4 -name "last.pt" 2>/dev/null
!ls -lh checkpoints/ experiments/ 2>&1 | head -n 20

/content/chess-transformer
/content/chess-transformer/checkpoints/best_model.pt
/content/chess-transformer/checkpoints/last.pt
checkpoints/:
total 404M
-rw-r--r-- 1 root root 101M Sep  8 17:21 best_model.pt
-rw-r--r-- 1 root root 303M Sep  8 20:15 last.pt

experiments/:
total 48K
-rw-r--r-- 1 root root  29K Sep  8 20:27 agent_log.jsonl
drwxr-xr-x 2 root root 4.0K Sep  8 17:03 configs
-rw-r--r-- 1 root root  107 Sep  8 20:19 evaluation_results.json
-rw-r--r-- 1 root root 1.8K Sep  8 20:15 training_log.jsonl
-rw-r--r-- 1 root root 2.2K Sep  8 20:15 training_results.json


In [11]:
from google.colab import files
files.download('/content/chess-transformer/checkpoints/best_model.pt')  # 101MB, el importante

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
!tar -czf /content/small.tar.gz -C /content/chess-transformer experiments/ paper/main.tex data/processed/stats.json data/processed/vocab.json
from google.colab import files
files.download('/content/small.tar.gz')  # KB, métricas + paper

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
from google.colab import files
files.download('/content/chess-transformer/checkpoints/last.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
!cat experiments/training_log.jsonl

{"epoch": 0, "loss": 7.698278828913393, "val_loss": 7.606123447418213, "lr": 0.0002967221401100708, "minutes": 12.6}
{"epoch": 1, "loss": 7.661302673051215, "val_loss": 7.96268064620647, "lr": 0.0002870318186463901, "minutes": 12.5}
{"epoch": 2, "loss": 7.652272670858903, "val_loss": 7.8085974794753055, "lr": 0.0002713525491562421, "minutes": 12.4}
{"epoch": 3, "loss": 7.647827798601861, "val_loss": 8.127071796579564, "lr": 0.00025036959095382875, "minutes": 12.4}
{"epoch": 4, "loss": 7.647076832429144, "val_loss": 7.915506253343947, "lr": 0.00022500000000000002, "minutes": 12.3}
{"epoch": 5, "loss": 7.641335288879847, "val_loss": 7.986418646954475, "lr": 0.00019635254915624213, "minutes": 12.4}
{"epoch": 6, "loss": 7.640662755996162, "val_loss": 7.883654580217726, "lr": 0.00016567926949014806, "minutes": 12.3}
{"epoch": 7, "loss": 7.638051445836383, "val_loss": 7.999691713617203, "lr": 0.00013432073050985205, "minutes": 12.3}
{"epoch": 8, "loss": 7.632700061177158, "val_loss": 8.10223

In [ ]:
# PILOTO masked-CE (3 epocas). Si CE<7.0 -> lanzar full.
from src.graph_agents.llm_agents import run_agent
r = run_agent('trainer', 'Masked-CE pilot', epochs=3, batch_size=128, lr=1e-3,
              warmup_ratio=0.05, mask_illegal=True, run_id='masked-v1',
              seed=42, resume=True, use_amp=True)
print('mode:', r['mode'])
print(r['result'])


In [ ]:
# FULL (continua desde el piloto via resume, mismo run_id)
from src.graph_agents.llm_agents import run_agent
r = run_agent('trainer', 'Masked-CE full', epochs=15, batch_size=128, lr=1e-3,
              warmup_ratio=0.05, mask_illegal=True, run_id='masked-v1',
              seed=42, resume=True, use_amp=True)
print('mode:', r['mode'])
print(r['result'])
